# 📘 W1 Python Lab — 가격 1차원 수집

**n8n W1 강의의 Python 버전.** 같은 일을 Python으로 해봅니다.

## 학습 목표
- `requests` 라이브러리로 외부 API 호출
- JSON 응답 파싱
- 3개 데이터 소스 통합 (Yahoo Finance · CoinGecko · FRED)
- 결과를 딕셔너리로 정리

## 🛠 사전 준비

```bash
pip install requests python-dotenv
```

`.env` 파일 생성:
```
FRED_API_KEY=여러분의_FRED_키
```

> 💡 **n8n과 비교:** n8n의 HTTP Request 노드 = Python의 `requests.get()`. Credentials = `.env` 파일. 개념은 완전히 동일합니다.

## 1. 필요한 라이브러리 import

In [ ]:
import requests
import os
from datetime import datetime
from dotenv import load_dotenv

# .env 파일에서 환경변수 로드
load_dotenv()

FRED_API_KEY = os.getenv('FRED_API_KEY')
print(f'FRED API Key loaded: {"✓" if FRED_API_KEY else "✗"}')

## 2. Yahoo Finance — 주식·지수 현재가

n8n에서는 `HTTP Request` 노드로 호출했습니다. Python에서도 동일합니다.

**중요:** User-Agent 헤더 필수.

In [ ]:
def get_yahoo_price(symbol: str) -> dict:
    """Yahoo Finance에서 주식/지수 현재가 조회."""
    url = f'https://query1.finance.yahoo.com/v8/finance/chart/{symbol}'
    headers = {
        'User-Agent': 'Mozilla/5.0 (Educational Project)'  # 필수
    }
    params = {'range': '1d', 'interval': '1m'}
    
    response = requests.get(url, headers=headers, params=params, timeout=10)
    response.raise_for_status()
    
    data = response.json()
    meta = data['chart']['result'][0]['meta']
    
    return {
        'symbol': meta['symbol'],
        'price': meta['regularMarketPrice'],
        'currency': meta['currency'],
        'exchange': meta.get('exchangeName', 'N/A')
    }

# 테스트: 애플 + S&P 500
aapl = get_yahoo_price('AAPL')
sp500 = get_yahoo_price('^GSPC')

print('AAPL:', aapl)
print('S&P 500:', sp500)

## 3. CoinGecko — 암호화폐 현재가

Demo API는 무료, 키 불필요, 30회/분 제한.

In [ ]:
def get_crypto_prices(coins: list[str]) -> dict:
    """CoinGecko에서 여러 코인 현재가 조회."""
    url = 'https://api.coingecko.com/api/v3/simple/price'
    params = {
        'ids': ','.join(coins),
        'vs_currencies': 'usd,krw',
        'include_24hr_change': 'true'
    }
    
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()

# 테스트: BTC + ETH
crypto = get_crypto_prices(['bitcoin', 'ethereum'])
print('Crypto prices:')
for coin, data in crypto.items():
    print(f'  {coin}: ${data["usd"]:,.2f} ({data["usd_24h_change"]:+.2f}% 24h)')

## 4. FRED — 거시 지표 (환율·VIX)

미 연준 데이터베이스. 무료 키 필요.  
발급: https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
def get_fred_latest(series_id: str) -> dict:
    """FRED에서 시리즈 최신값 조회.
    
    주요 시리즈:
    - DEXKOUS: 원/달러 환율
    - VIXCLS: VIX 지수
    - DGS10: 미국 10년물 국채금리
    - CPIAUCSL: 미국 CPI
    """
    if not FRED_API_KEY:
        raise ValueError('FRED_API_KEY가 .env에 없습니다')
    
    url = 'https://api.stlouisfed.org/fred/series/observations'
    params = {
        'series_id': series_id,
        'api_key': FRED_API_KEY,
        'file_type': 'json',
        'sort_order': 'desc',
        'limit': 1
    }
    
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    
    latest = response.json()['observations'][0]
    return {
        'series': series_id,
        'date': latest['date'],
        'value': float(latest['value']) if latest['value'] != '.' else None
    }

# 테스트: 원/달러 + VIX
usdkrw = get_fred_latest('DEXKOUS')
vix = get_fred_latest('VIXCLS')

print(f'USD/KRW: {usdkrw["value"]} ({usdkrw["date"]})')
print(f'VIX: {vix["value"]} ({vix["date"]})')

## 5. 통합 — 하나의 스냅샷으로

n8n의 `Set` 노드 역할. 3개 API 결과를 하나의 딕셔너리로 병합.

In [ ]:
def daily_snapshot() -> dict:
    """매일 실행할 시장 스냅샷 생성 (n8n 전체 워크플로와 동일)."""
    snapshot = {
        'timestamp': datetime.now().isoformat(),
        'stocks': {},
        'crypto': {},
        'macro': {}
    }
    
    # 1. 주식
    for ticker in ['AAPL', '^GSPC']:
        try:
            data = get_yahoo_price(ticker)
            snapshot['stocks'][ticker] = data['price']
        except Exception as e:
            snapshot['stocks'][ticker] = None
            print(f'Error {ticker}: {e}')
    
    # 2. 암호화폐
    try:
        crypto = get_crypto_prices(['bitcoin', 'ethereum'])
        snapshot['crypto'] = {
            'BTC_USD': crypto['bitcoin']['usd'],
            'BTC_KRW': crypto['bitcoin']['krw'],
            'ETH_USD': crypto['ethereum']['usd']
        }
    except Exception as e:
        print(f'Crypto error: {e}')
    
    # 3. 매크로 (FRED 키 있을 때만)
    if FRED_API_KEY:
        for series in ['DEXKOUS', 'VIXCLS']:
            try:
                data = get_fred_latest(series)
                snapshot['macro'][series] = data['value']
            except Exception as e:
                print(f'FRED {series} error: {e}')
    
    return snapshot

# 실행
snap = daily_snapshot()
import json
print(json.dumps(snap, indent=2, ensure_ascii=False))

## 6. CSV로 저장 — Sheets 대신

n8n에서는 Google Sheets에 저장했습니다. Python에서는 `pandas`로 CSV에 누적.

In [ ]:
import pandas as pd
from pathlib import Path

def append_to_csv(snapshot: dict, filepath: str = 'daily_snapshots.csv'):
    """스냅샷을 CSV에 추가 (없으면 생성)."""
    # flatten for CSV
    row = {'timestamp': snapshot['timestamp']}
    for ticker, price in snapshot.get('stocks', {}).items():
        row[f'stock_{ticker}'] = price
    for coin, price in snapshot.get('crypto', {}).items():
        row[f'crypto_{coin}'] = price
    for series, value in snapshot.get('macro', {}).items():
        row[f'macro_{series}'] = value
    
    df_new = pd.DataFrame([row])
    path = Path(filepath)
    
    if path.exists():
        df_new.to_csv(path, mode='a', header=False, index=False)
        print(f'✓ Appended to {filepath}')
    else:
        df_new.to_csv(path, index=False)
        print(f'✓ Created {filepath}')
    
    # 최근 5개 읽어서 보여주기
    df = pd.read_csv(path)
    return df.tail(5)

# 실행 — 몇 번 반복해서 CSV 쌓기
append_to_csv(snap)

## 🎯 과제

1. **관심 종목 3개 추가** — 본인 관심 종목(TSLA, NVDA, 005930.KS 등)을 `daily_snapshot()`에 추가
2. **FRED 시리즈 확장** — 한국 금리(IR3TIB01KRM156N), 한국 CPI(KORCPIALLMINMEI) 추가
3. **에러 핸들링** — 한 API가 실패해도 다른 API는 계속 실행되도록 `try-except` 강화
4. **스케줄러 연결** — `schedule` 라이브러리로 매일 오전 9시 자동 실행 구현
   ```python
   pip install schedule
   import schedule
   schedule.every().day.at('09:00').do(lambda: append_to_csv(daily_snapshot()))
   ```

## 🔄 n8n vs Python 비교

| 개념 | n8n | Python |
|---|---|---|
| HTTP 호출 | HTTP Request 노드 | `requests.get()` |
| 스케줄 | Schedule Trigger | `schedule` 라이브러리 or cron |
| 데이터 병합 | Merge/Set 노드 | 딕셔너리 병합 |
| 저장소 | Google Sheets 노드 | pandas `to_csv()` |
| 자격 증명 | Credentials | `.env` 파일 |

**핵심:** 개념은 같고, 구현만 다릅니다. n8n이 시각적이라면 Python은 코드로 세밀한 제어. 상황에 맞게 선택.